# Wednesday, hands on: the protect list, the flag and the plan line
#
# > "Ties matter. If two members spent the same, I want them ranked the same, and I want to know
# > how many made the top fifty, not forty-nine because of a tie."
#
# Replace every `__TODO__`. One of these steps has a business answer rather than a technical one,
# and the checks will not tell you which.

In [ ]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
kit.flow(["rank", "meet the tie", "choose a rule", "flag the fallers", "accumulate"],
         lit=[0], title="Where you are")

## 1. Q2 revenue per Retail-Plus member
#
# One row per member, carrying their Q2 revenue. This is still `GROUP BY` territory.

In [ ]:
q2 = kit.sql("__TODO1__", conn=conn)
kit.check("more than fifty members to rank", len(q2) > 50, f"{len(q2)} members")

## 2. Three rankings side by side
#
# Add `row_number`, `rank` and `dense_rank` over the same ordering. Look at positions 46 to 54
# rather than the top ten; the top of a list is never where the argument happens.

In [ ]:
edge = kit.sql("__TODO2__", conn=conn)
kit.table(list(edge[0]) if edge else ["rn"], [list(r.values()) for r in edge],
          caption="The boundary, where the tie lives")
kit.check("nine rows around the boundary", len(edge) == 9, f"{len(edge)} rows")

## 3. How many names does each rule ship?
#
# Count how many rows survive a filter at fifty under each of the three functions.

In [ ]:
counts = kit.sql("__TODO3__", conn=conn)[0]
print(dict(counts))
kit.check("the three rules disagree", len(set(counts.values())) == 3, str(dict(counts)))
kit.decision_ladder(["ROW_NUMBER, which drops one of a tied pair",
                     "DENSE_RANK, which moves the cut further down",
                     "RANK, which keeps both tied members"],
                    cut_at=2, title="Which one he asked for")

## 4. The protect list
#
# Top fifty per segment under the rule the head of Retail-Plus asked for. Remember that a window
# cannot be filtered in `WHERE`, so compute it inside and filter outside.

In [ ]:
protect = kit.sql("__TODO4__", conn=conn)
kit.check("Retail-Plus ships fifty-one names",
          next((r for r in protect if r["segment"] == "Retail-Plus"))["names"] == 51, str(protect))

## 5. Falling two months running
#
# Monthly spend per member, then two LAGs and a comparison. Decide before you write it what a
# member with only one month of data should do to your flag.

In [ ]:
falling = kit.sql("__TODO5__", conn=conn)
kit.check("three members fall in both steps", len(falling) == 3, f"{len(falling)}")
kit.vflow(["July", "August", "September"], lit=[2], title="Two steps down, not one")

## 6. The running total against plan
#
# Weekly Q2 revenue, accumulating, beside the plan line accumulating. Give the window an order
# that cannot tie, or the cumulative column can differ between runs.

In [ ]:
plan = kit.sql("__TODO6__", conn=conn)
kit.check("thirteen or fourteen weeks in the quarter", 13 <= len(plan) <= 14, f"{len(plan)}")
kit.ladder(["weekly revenue", "cumulative", "plan cumulative", "the gap between them"],
           lit=[3], title="What Meera reads")

## 7. The note
#
# Write four sentences: which tie rule you chose, the sentence from the head of Retail-Plus that
# decided it, how many names the Retail-Plus list contains, and what you would say to a flagged
# member who was on holiday in August.
#
# The fourth sentence is the one that matters. A flag is a shortlist for a conversation rather
# than a verdict.

In [ ]:
kit.matrix(["the list", "the flag", "the plan line"],
           ["what it is", "what it is not"],
           [["fifty-one names", "a ranking of worth"],
            ["three members to call", "proof of anything"],
            ["ahead, then level", "a forecast"]],
           title="What you are handing over")
kit.check_summary()